In [12]:
import pandas as pd
import os
import shutil
from subprocess import run

In [15]:
mastersheetFile = '/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI/scripts/fMRI/master_subject_10stories_highacc.csv'
df_master = pd.read_csv(mastersheetFile)
df_master.loc[df_master['transcript'].isin(['prettymouth','milkywayoriginal','milkywayvodka','slumlordreach','21styear']),'protocol'] = 'skyra'
df_master.loc[df_master['transcript'].isin(['shapessocial']),'protocol'] = 'Prisma_MB4'
df_master.loc[df_master['transcript'].isin(['piemanpni','bronx','black','forgot']),'protocol'] = 'Prisma_MB3'

# Center age at the mean of the 102 unique participants, so participants with
# multiple scans do not contribute multiple times to the reference age.
df_master['age'] = pd.to_numeric(df_master['age'], errors='raise')
age_reference = df_master.groupby('subID')['age'].mean().mean()
df_master['age_c'] = df_master['age'] - age_reference

# AFNI will treat sex as categorical because it is not listed in -qVars.
assert set(df_master['sex'].dropna().unique()) == {'F', 'M'}
print(f'Age reference (unique-participant mean): {age_reference:.3f} years')
df_master

Age reference (unique-participant mean): 22.269 years


,subID,task,age,sex,condition,comprehension,transcript,label,protocol,age_c
0,sub-023,prettymouth,28,F,affair,0.889,prettymouth,prettymouth,skyra,5.730719
1,sub-023,milkyway,28,F,vodka,1.000,milkywayvodka,milkywayvodka,skyra,5.730719
2,sub-030,prettymouth,21,F,paranoia,0.963,prettymouth,prettymouth,skyra,-1.269281
3,sub-030,milkyway,21,F,vodka,0.893,milkywayvodka,milkywayvodka,skyra,-1.269281
4,sub-032,prettymouth,22,M,affair,0.963,prettymouth,prettymouth,skyra,-0.269281
...,...,...,...,...,...,...,...,...,...,...
208,sub-314,piemanpni,25,F,NaN,0.767,piemanpni,piemanpni,Prisma_MB3,2.730719
209,sub-314,bronx,25,F,NaN,0.780,bronx,bronx,Prisma_MB3,2.730719
210,sub-314,forgot,25,F,NaN,0.780,forgot,forgot,Prisma_MB3,2.730719
211,sub-314,black,25,F,NaN,0.920,black,black,Prisma_MB3,2.730719


In [28]:
df_master.loc[df_master['subID'].isin(['sub-235','sub-190','sub-267','sub-265'])]

,subID,task,age,sex,condition,comprehension,transcript,label,protocol
76,sub-190,shapessocial,21,F,social-movie-physical,1.000,shapessocial,shapessocial,Prisma_MB4
77,sub-190,21styear,21,F,NaN,1.000,21styear,21styear,skyra
90,sub-235,shapessocial,19,F,social-physical-movie,1.000,shapessocial,shapessocial,Prisma_MB4
91,sub-235,21styear,19,F,NaN,0.889,21styear,21styear,skyra
96,sub-265,21styear,19,F,NaN,0.926,21styear,21styear,skyra
97,sub-265,piemanpni,20,F,NaN,0.600,piemanpni,piemanpni,Prisma_MB3
98,sub-265,bronx,20,F,NaN,0.640,bronx,bronx,Prisma_MB3
99,sub-265,forgot,20,F,NaN,0.640,forgot,forgot,Prisma_MB3
100,sub-265,black,20,F,NaN,0.760,black,black,Prisma_MB3
101,sub-267,21styear,26,F,NaN,0.963,21styear,21styear,skyra


In [27]:
df_tmp = df_master.groupby(['subID','protocol']).count().reset_index().groupby(['subID']).count()
df_tmp.sort_values('protocol')

,protocol,task,age,sex,condition,comprehension,transcript,label
subID,,,,,,,,
sub-023,1,1,1,1,1,1,1,1
sub-030,1,1,1,1,1,1,1,1
sub-032,1,1,1,1,1,1,1,1
sub-034,1,1,1,1,1,1,1,1
sub-049,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...
sub-315,1,1,1,1,1,1,1,1
sub-235,2,2,2,2,2,2,2,2
sub-190,2,2,2,2,2,2,2,2


In [13]:
Dir_working = '/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis'
Nvar = 113
NFA = 8
flag_model = f'Nvar{Nvar}NFA{NFA}_LPAC_multipleReg_unsmoothed'

# Dir_figures = f'/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/{flag_model}/figures_SVC/'
# os.makedirs(Dir_figures,exist_ok=True)

# groupMask = '/work/desai-lab/xuanyang/Project/dataset/Nastase/narratives/derivatives/afni-nosmooth/tpl-MNI152NLin2009cAsym/nosmooth_mask_allstories.nii'
# filename = 'GMxrs_mask_SVC2_LR_33.nii.gz'
# filename = 'rs_mask_SVC_LR_33.nii.gz'
filename = 'rs_mask_GM_33.nii.gz'
Dir_masks = '/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/fMRI/masks'
groupMask = os.path.join(Dir_masks,filename)

# folder = 'threshold_SVC2_LR'
folder = 'nonthreshold_LMEr_r1.3'
atlas_name = 'HCPex'
Dir_model = os.path.join(Dir_working,'models',flag_model,f'FA{NFA}_{atlas_name}')
Dir_results = os.path.join(Dir_model,'results')
Dir_results_1st = os.path.join(Dir_results,'firstlevel')
Dir_results_2nd = os.path.join(Dir_results,'secondlevel')

In [16]:
for iFA in range(1,NFA+1):
    List_vars = [f'FA{iFA}_n{NFA}']
    for i,subID, task,label in zip(df_master.index,df_master.subID,df_master.task,df_master.label):
        idx = "{:03d}".format(i+1)
        Dir_sub_task = os.path.join(Dir_results_1st,subID,task)
        path_img_stats = os.path.join(Dir_sub_task,f'{subID}_{flag_model}_{atlas_name}_FA{iFA}.nii.gz')

        df_master.loc[i,'finished'] = 0

        if os.path.exists(path_img_stats):
            df_master.loc[i,'finished'] = 1
            df_master.loc[i,'img_stats'] = path_img_stats
    

    subfolder = f'{len(df_master)}highacc'
    subList_ana_cmplt = df_master.loc[df_master.finished==1].reset_index(drop=True)
    Nsub = len(df_master)
    

    resultDir_2nd = os.path.join(Dir_results_2nd,folder)
    masterScript = os.path.join(resultDir_2nd,f'run_secondlevel_{subfolder}.sh')
    if not os.path.exists(resultDir_2nd):
        os.makedirs(resultDir_2nd)
    os.chdir(resultDir_2nd)
    
    df_lme_table = subList_ana_cmplt[['subID','transcript','age_c','sex']].copy()
    df_lme_table.rename({'subID':'Subj'},axis=1,inplace=True)
    df_lme_table['InputFile'] = [f"{statImg}" for statImg in subList_ana_cmplt['img_stats']]
    df_lme_table.to_csv(os.path.join(resultDir_2nd,f"dataTable_FA{iFA}.txt"),index=False,sep='\t')



    scriptFile = os.path.join(resultDir_2nd,f'Batch_secondlevel_FA{iFA}')
    transcript_glt = (
        "transcript : "
        "0.046948*21styear "
        "+0.187793*black "
        "+0.107981*bronx "
        "+0.107981*forgot "
        "+0.065728*milkywayoriginal "
        "+0.079812*milkywayvodka "
        "+0.140845*piemanpni "
        "+0.164319*prettymouth "
        "+0.084507*shapessocial "
        "+0.014085*slumlordreach"
    )

    with open(scriptFile, "w") as f:
        f.write("#!/bin/tcsh -xef \n")
        f.write(f"/work/apps/AFNI/26.0.08/3dLMEr -prefix {f'FA{iFA}'} \\\n")
        f.write(f"-resid FA{iFA}_resid \\\n")
        f.write(f"-mask {groupMask} \\\n")
        f.write("-model 'transcript+age_c+sex+(1|Subj)' \\\n")
        f.write("-qVars 'age_c' \\\n")
        f.write("-IF InputFile \\\n")
        f.write("-SS_type 3 \\\n")
        f.write(f"-gltCode mean '{transcript_glt}' \\\n")
        f.write("-gltCode age 'age_c :' \\\n")
        f.write("-gltCode male_vs_female 'sex : 1*M -1*F' \\\n")
        f.write(f"-dataTable @/{resultDir_2nd}/dataTable_FA{iFA}.txt")
        
    os.chdir(resultDir_2nd)
    # run(f"tcsh {scriptFile}",shell=True)

In [17]:
resultDir_2nd

'/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA8_LPAC_multipleReg_unsmoothed/FA8_HCPex/results/secondlevel/nonthreshold_LMEr_r1.3'

# Estimate ACF

In [11]:
Dir_results_2nd

'/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA8_LPAC_multipleReg_unsmoothed/FA8_HCPex/results/secondlevel'

# 3dClustSim

In [12]:
# Configure and parse the residual ACF estimates produced by 3dFWHMx.
# ACF parameters describe the spatial correlation of the model residuals.
from pathlib import Path
import math
import re
import subprocess

clustsim_dir = Path(Dir_results_2nd) / folder
clustsim_dir.mkdir(parents=True, exist_ok=True)
afni_bin = Path('/work/apps/AFNI/26.0.08')
voxel_p = 0.001       # total two-sided voxelwise p-value
cluster_alpha = 0.05  # cluster-level FWE within each factor map
NN = 2                # face + edge neighbors
number_pattern = re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[Ee][-+]?\d+)?')

def numeric_values(text):
    return [float(value) for value in number_pattern.findall(text)]

def read_acf(acf_file):
    candidates = []
    for raw_line in Path(acf_file).read_text(errors='replace').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        values = numeric_values(line)
        if len(values) >= 3:
            a, b, c = values[:3]
            if 0 <= a <= 1 and b > 0 and c > 0:
                candidates.append((a, b, c))
    if not candidates:
        raise ValueError(f'No valid ACF triplet found in {acf_file}')
    return candidates[-1]

def read_cluster_cutoff(table_file, target_p=0.001, target_alpha=0.05):
    lines = Path(table_file).read_text(errors='replace').splitlines()
    alpha_values = None
    for index, line in enumerate(lines):
        if 'pthr' in line.lower() and 'alpha' in line.lower():
            for candidate in lines[index:index + 4]:
                if '|' in candidate:
                    values = numeric_values(candidate.split('|', 1)[1])
                    if values:
                        alpha_values = values
                        break
            if alpha_values:
                break
    if not alpha_values:
        raise ValueError(f'Could not read alpha columns from {table_file}')
    alpha_index = min(range(len(alpha_values)), key=lambda i: abs(alpha_values[i] - target_alpha))
    if not math.isclose(alpha_values[alpha_index], target_alpha, abs_tol=5e-7):
        raise ValueError(f'alpha={target_alpha} is absent from {table_file}')
    rows = []
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        values = numeric_values(stripped)
        if len(values) >= 1 + len(alpha_values):
            rows.append((values[0], values[1:1 + len(alpha_values)]))
    if not rows:
        raise ValueError(f'Could not read p-threshold rows from {table_file}')
    row_p, cutoffs = min(rows, key=lambda row: abs(row[0] - target_p))
    if not math.isclose(row_p, target_p, abs_tol=5e-7):
        raise ValueError(f'p={target_p} is absent from {table_file}')
    raw_cutoff = cutoffs[alpha_index]
    return raw_cutoff, math.ceil(raw_cutoff)


In [15]:
# Generate and run one auditable 3dClustSim Bash script for each factor.
# Existing simulation tables are reused so an accidental notebook rerun is inexpensive.
clustsim_rows = []
for iFA in range(1, NFA + 1):
    prefix = f'FA{iFA}'
    acf_file = clustsim_dir / f'{prefix}_ACF.txt'
    if not acf_file.exists():
        raise FileNotFoundError(f'Missing ACF output: {acf_file}')

    acf_a, acf_b, acf_c = read_acf(acf_file)
    sim_prefix = clustsim_dir / f'{prefix}.CSimA'
    sim_table = Path(f'{sim_prefix}.NN{NN}_bisided.1D')
    bash_file = clustsim_dir / f'Batch_3dClustSim_{prefix}.sh'
    bash_text = f'''#!/usr/bin/env bash
set -euo pipefail
cd {clustsim_dir}
{afni_bin / '3dClustSim'} \
    -mask {groupMask} \
    -acf {acf_a:.10g} {acf_b:.10g} {acf_c:.10g} \
    -pthr {voxel_p:.10g} \
    -athr {cluster_alpha:.10g} \
    -LOTS \
    -prefix {sim_prefix}
'''
    bash_file.write_text(bash_text)
    bash_file.chmod(0o755)

    if not sim_table.exists():
        print(f'Running {bash_file.name}')
        subprocess.run(['bash', str(bash_file)], cwd=clustsim_dir, check=True)
    else:
        print(f'Reusing {sim_table.name}')

    raw_nvox, applied_nvox = read_cluster_cutoff(sim_table, voxel_p, cluster_alpha)
    clustsim_rows.append({
        'FA': iFA, 'acf_a': acf_a, 'acf_b': acf_b, 'acf_c': acf_c,
        'voxel_p_bisided': voxel_p, 'cluster_alpha': cluster_alpha, 'NN': NN,
        'raw_cluster_nvox': raw_nvox, 'applied_cluster_nvox': applied_nvox,
        'table': sim_table.name,
    })

df_clustsim = pd.DataFrame(clustsim_rows)
df_clustsim.to_csv(clustsim_dir / 'LMEr_3dClustSim_thresholds.csv', index=False)
df_clustsim


Running Batch_3dClustSim_FA1.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 46.97 mm
++ ACF(0.16,22.83,6.75) => FWHM=11.98 => 65x77x49 pads to 120x120x80
 + Kernel image dimensions 59 x 59 x 39
++ Startup clock time = 0.2 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 321.9 s


Running Batch_3dClustSim_FA2.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 45.01 mm
++ ACF(0.15,22.12,6.77) => FWHM=11.80 => 65x77x49 pads to 120x120x80
 + Kernel image dimensions 59 x 59 x 39
++ Startup clock time = 0.1 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 323.8 s


Running Batch_3dClustSim_FA3.sh


...

Running Batch_3dClustSim_FA7.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 44.49 mm
++ ACF(0.15,21.74,6.70) => FWHM=11.76 => 65x77x49 pads to 96x120x80
 + Kernel image dimensions 47 x 59 x 39
++ Startup clock time = 0.2 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 240.2 s


Running Batch_3dClustSim_FA8.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 43.07 mm
++ ACF(0.16,20.86,6.68) => FWHM=11.79 => 65x77x49 pads to 96x120x80
 + Kernel image dimensions 47 x 59 x 39
++ Startup clock time = 0.1 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 229.8 s


,FA,acf_a,acf_b,acf_c,voxel_p_bisided,cluster_alpha,NN,raw_cluster_nvox,applied_cluster_nvox,table
0,1,0.159466,22.8267,6.74620,0.001,0.05,2,18.6,19,FA1.CSimA.NN2_bisided.1D
1,2,0.149868,22.1186,6.76584,0.001,0.05,2,18.7,19,FA2.CSimA.NN2_bisided.1D
2,3,0.156225,23.0886,6.74251,0.001,0.05,2,18.7,19,FA3.CSimA.NN2_bisided.1D
3,4,0.166546,22.5265,6.67198,0.001,0.05,2,18.9,19,FA4.CSimA.NN2_bisided.1D
4,5,0.159914,23.0173,6.74848,0.001,0.05,2,18.6,19,FA5.CSimA.NN2_bisided.1D
5,6,0.158536,22.5777,6.68807,0.001,0.05,2,18.4,19,FA6.CSimA.NN2_bisided.1D
6,7,0.153407,21.7376,6.69960,0.001,0.05,2,18.7,19,FA7.CSimA.NN2_bisided.1D
7,8,0.157197,20.8630,6.68434,0.001,0.05,2,18.6,19,FA8.CSimA.NN2_bisided.1D
